<a href="https://colab.research.google.com/github/mshitie/quantitative_XAI/blob/main/BUSBRA_table_XI_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BUS-BRA: patient-level train / validation / test -> Table XI

Trains ResNet50, EfficientNetB0 and DenseNet121 on a patient-grouped training partition of BUS-BRA, selects checkpoints on a validation partition, and reports **Table XI** on a held-out test partition of patients (cluster bootstrap over patients for the 95% CIs).

Because training and test data come from the same dataset, this is an **internal held-out evaluation, not an external validation**.

Upload `BUSBRA.zip` to `/content/` and use a GPU runtime. Run the cells in order.

In [1]:
import zipfile

zip_path = '/content/BUSBRA.zip'
extract_path = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipping completed successfully!")

Unzipping completed successfully!


In [3]:
# 1. Setup and configuration
import os, re, glob, gc, warnings, random
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import ResNet50, EfficientNetB0, DenseNet121
from tensorflow.keras.applications import resnet50, efficientnet, densenet
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, roc_auc_score
from google.colab import drive

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError: pass
#drive.mount("/content/drive")

BUSBRA_DIR = "/content/BUSBRA"
IMAGE_DIR, CSV_FILE = f"{BUSBRA_DIR}/Images", f"{BUSBRA_DIR}/bus_data.csv"
MODEL_DIR = "/content/drive/MyDrive/BUSBRA_trained_models"      # checkpoints survive runtime resets
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE, BATCH = 320, 16
EPOCHS_HEAD, EPOCHS_FT, LR_HEAD, LR_FT = 10, 40, 1e-3, 1e-4
DROPOUT, DENSE_UNITS, TRAIN_BN = 0.5, 256, True
BASE_TRAINING = False            # standard inference-mode BatchNorm for the frozen parts
CLASSES = ["benign", "malignant"]
N_BOOT = 2000
BACKBONES = {
    "ResNet50": (ResNet50, resnet50.preprocess_input, ("conv3_", "conv4_", "conv5_")),
    "EfficientNetB0": (EfficientNetB0, efficientnet.preprocess_input, ("block4", "block5", "block6", "block7", "top_")),
    "DenseNet121": (DenseNet121, densenet.preprocess_input, ("conv3_block", "conv4_block", "conv5_block", "pool3_", "pool4_", "bn")),
}
MODEL_PATHS = {n: f"{MODEL_DIR}/{n.lower()}_busbra_v2.keras" for n in BACKBONES}
print("GPUs:", tf.config.list_physical_devices("GPU"))

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# 2. BUS-BRA manifest and patient-level split (no patient appears in more than one partition)
def norm_id(v):
    return re.sub(r"\.(png|jpe?g|bmp|tiff?)$", "", str(v).strip().replace("\\", "/").split("/")[-1], flags=re.I)

meta = pd.read_csv(CSV_FILE)
def find_col(cands):
    low = {str(c).strip().lower(): c for c in meta.columns}
    return next((low[c] for c in cands if c in low), None)

id_col  = find_col(["id", "image_id", "image", "filename", "file_name", "image_name"])
lab_col = find_col(["pathology", "diagnosis", "label", "class", "classification"])
pat_col = find_col(["case", "patient_id", "patient", "patientid", "subject_id", "case_id", "subject"])
assert id_col and lab_col, f"Cannot identify ID/label columns in {meta.columns.tolist()}"
print("columns used -> id:", id_col, "| label:", lab_col, "| patient:", pat_col)
if pat_col is None:
    print("WARNING: no patient column; patient inferred from the leading number in the image ID. Verify before reporting.")

rows = []
for _, r in meta.iterrows():
    stem, label = norm_id(r[id_col]), str(r[lab_col]).strip().lower()
    imgs = sorted(glob.glob(os.path.join(IMAGE_DIR, stem + ".*")))
    if label not in CLASSES or not imgs:
        continue
    patient = str(r[pat_col]).strip() if pat_col else re.search(r"\d+", stem).group()
    rows.append(dict(img_id=stem, path=imgs[0], label=label, patient=patient))

df = pd.DataFrame(rows).drop_duplicates("img_id").sort_values("img_id").reset_index(drop=True)
assert len(df), "No BUS-BRA images matched; check CSV columns and file names."
df["y"] = df.label.map({"benign": 0, "malignant": 1}).astype(int)
print(f"Matched {len(df)} images / {df.patient.nunique()} patients (dataset description: 1,875 / 1,064)")

# 5-fold patient-grouped outer split: fold 0 = test; 4-fold inner split of the rest: fold 0 = validation
tv, te = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(df, df.y, df.patient))
trainval, ext = df.iloc[tv].reset_index(drop=True), df.iloc[te].reset_index(drop=True)
a, b = next(StratifiedGroupKFold(4, shuffle=True, random_state=SEED + 1).split(trainval, trainval.y, trainval.patient))
train_df, val_df = trainval.iloc[a].reset_index(drop=True), trainval.iloc[b].reset_index(drop=True)
for p, q in [(train_df, val_df), (train_df, ext), (val_df, ext)]:
    assert not set(p.patient) & set(q.patient), "patient leakage between partitions"
for n, f in [("train", train_df), ("validation", val_df), ("test", ext)]:
    print(f"{n}: {len(f)} images / {f.patient.nunique()} patients | {f.label.value_counts().to_dict()}")
N_IMG, N_PAT = len(ext), ext.patient.nunique()
pd.concat([train_df.assign(split="train"), val_df.assign(split="validation"), ext.assign(split="test")]
          ).to_csv(f"{MODEL_DIR}/BUSBRA_patient_splits.csv", index=False)

cw = compute_class_weight("balanced", classes=np.unique(train_df.y), y=train_df.y)
CLASS_WEIGHTS = {int(c): float(w) for c, w in zip(np.unique(train_df.y), cw)}
print("class weights:", CLASS_WEIGHTS)

columns used -> id: ID | label: Pathology | patient: Case
Matched 1875 images / 1064 patients (dataset description: 1,875 / 1,064)
train: 1128 images / 638 patients | {'benign': 775, 'malignant': 353}
validation: 373 images / 213 patients | {'benign': 255, 'malignant': 118}
test: 374 images / 213 patients | {'benign': 238, 'malignant': 136}
class weights: {0: 0.727741935483871, 1: 1.5977337110481586}


In [5]:
# 3. Train the three backbones on the BUS-BRA training partition (checkpoint = best validation loss)
def make_ds(frame, prep, training):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.astype(str).to_numpy(), frame.y.astype(np.int32).to_numpy()))
    if training: ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def load(p, l):
        im = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        im.set_shape([None, None, 3])
        im = tf.image.resize_with_pad(tf.cast(im, tf.float32), IMG_SIZE, IMG_SIZE)     # keeps aspect ratio
        if training:                                                                    # augmentation (train only)
            im = tf.image.random_flip_left_right(im)
            im = tf.image.random_brightness(im, 25.0)
            im = tf.image.random_contrast(im, 0.8, 1.2)
            c = tf.cast(IMG_SIZE * tf.random.uniform([], 0.85, 1.0), tf.int32)         # random crop, resize back
            im = tf.image.resize(tf.image.random_crop(im, [c, c, 3]), (IMG_SIZE, IMG_SIZE))
            im = tf.clip_by_value(im, 0.0, 255.0)
        return prep(im), l
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

def compile_(model, lr):
    model.compile(optimizer=optimizers.Adam(lr), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])

def fit(model, tr_ds, va_ds, epochs, patience, wpath):
    cbs = [callbacks.EarlyStopping("val_loss", patience=patience, verbose=1),
           callbacks.ModelCheckpoint(wpath, monitor="val_loss", save_best_only=True, save_weights_only=True)]
    model.fit(tr_ds, validation_data=va_ds, epochs=epochs, class_weight=CLASS_WEIGHTS, callbacks=cbs, verbose=2)
    model.load_weights(wpath)

def train(name):
    cls, prep, prefixes = BACKBONES[name]
    base = cls(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(base(inp, training=BASE_TRAINING))
    x = layers.Dropout(DROPOUT, name="dropout")(layers.Dense(DENSE_UNITS, activation="relu", name="feature_dense")(x))
    model = models.Model(inp, layers.Softmax(name="predictions")(layers.Dense(len(CLASSES), name="logits")(x)), name=name)
    tr_ds, va_ds = make_ds(train_df, prep, True), make_ds(val_df, prep, False)
    w1, w2 = f"{MODEL_DIR}/{name}_head_v2.weights.h5", f"{MODEL_DIR}/{name}_ft_v2.weights.h5"

    base.trainable = False; compile_(model, LR_HEAD)                  # phase 1: head only
    fit(model, tr_ds, va_ds, EPOCHS_HEAD, 5, w1)

    base.trainable = True                                              # phase 2: later blocks
    for l in base.layers:
        sel = l.name.startswith(prefixes)
        l.trainable = bool(sel and TRAIN_BN) if isinstance(l, layers.BatchNormalization) else bool(sel)
    steps = int(np.ceil(len(train_df) / BATCH))
    compile_(model, optimizers.schedules.CosineDecay(LR_FT, EPOCHS_FT * steps, alpha=0.01))
    fit(model, tr_ds, va_ds, EPOCHS_FT, 8, w2)

    tra = model.evaluate(make_ds(train_df, prep, False), verbose=0)[1]; vaa = model.evaluate(va_ds, verbose=0)[1]
    print(f"{name}: train acc {tra:.3f} | validation acc {vaa:.3f}  (big gap = overfitting; both low = underfitting)")
    model.save(MODEL_PATHS[name]); print("saved", MODEL_PATHS[name])
    del model, base, tr_ds, va_ds; gc.collect(); tf.keras.backend.clear_session()

for n in BACKBONES:
    if os.path.isfile(MODEL_PATHS[n]): print("exists, skipping", n); continue
    print("\n" + "=" * 60 + "\nTRAINING", n); train(n)
for n, p in MODEL_PATHS.items():
    m = tf.keras.models.load_model(p, compile=False)
    assert m.output_shape[-1] == 2; print(n, "OK"); del m


TRAINING ResNet50
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/10
71/71 - 66s - 927ms/step - accuracy: 0.5683 - loss: 0.8732 - val_accuracy: 0.7587 - val_loss: 0.5594
Epoch 2/10
71/71 - 10s - 143ms/step - accuracy: 0.6285 - loss: 0.6422 - val_accuracy: 0.7292 - val_loss: 0.5872
Epoch 3/10
71/71 - 10s - 145ms/step - accuracy: 0.6587 - loss: 0.6165 - val_accuracy: 0.7292 - val_loss: 0.5873
Epoch 4/10
71/71 - 10s - 138ms/step - accuracy: 0.6365 - loss: 0.6212 - val_accuracy: 0.7775 - val_loss: 0.5229
Epoch 5/10
71/71 - 10s - 145ms/step - accuracy: 0.6631 - loss: 0.6049 - val_accuracy: 0.5550 - val_loss: 0.7050
Epoch 6/10
71/71 - 10s - 145ms/step - accuracy: 0.6879 - loss: 0.5972 - val_accuracy: 0.7373 - val_loss: 0.5431
Epoch 7/10
71/71 - 11s - 152ms/step - accuracy: 0.7101 - loss: 0.5794 - val_accuracy: 0.7748 - val_loss: 0.5370
Epoch 8/10
71/71 - 12s - 168ms/step - accuracy: 0.7083 - loss: 0.5716 - val_accuracy: 0.6676 - val_loss: 0.6103
Epoch 9/10
71/71 - 10s - 144ms/ste

In [6]:
# 4. Held-out test evaluation (all test images) with patient-cluster bootstrap CIs
pats = ext.patient.to_numpy(); uniq = np.unique(pats)
rows_of = {p: np.where(pats == p)[0] for p in uniq}
_rng = np.random.RandomState(SEED)
BOOT = [np.concatenate([rows_of[p] for p in _rng.choice(uniq, len(uniq))]) for _ in range(N_BOOT)]
y = ext.y.to_numpy()
clf_rows = []

for name, (_, prep, _) in BACKBONES.items():
    m = tf.keras.models.load_model(MODEL_PATHS[name], compile=False)
    probs = m.predict(make_ds(ext, prep, False), verbose=0)      # 0 benign, 1 malignant; order matches ext
    assert probs.shape == (N_IMG, 2) and np.allclose(probs.sum(1), 1, atol=1e-3)
    pred = probs.argmax(1); correct = pred == y

    def stats(idx):
        return (correct[idx].mean(), f1_score(y[idx], pred[idx], labels=[0, 1], average="macro", zero_division=0),
                roc_auc_score(y[idx], probs[idx, 1]))
    point = stats(np.arange(N_IMG))
    boot = np.array([stats(i) for i in BOOT])
    lo, hi = np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
    clf_rows.append(dict(backbone=name, n_images=N_IMG, n_patients=N_PAT,
        accuracy=point[0], accuracy_ci_low=lo[0], accuracy_ci_high=hi[0],
        macro_f1=point[1], macro_f1_ci_low=lo[1], macro_f1_ci_high=hi[1],
        auc=point[2], auc_ci_low=lo[2], auc_ci_high=hi[2],
        misclassified=int((~correct).sum())))
    print(clf_rows[-1])
    del m; gc.collect(); tf.keras.backend.clear_session()

clf = pd.DataFrame(clf_rows)
clf.to_csv("/content/BUSBRA_table_xi.csv", index=False)
clf.round(4)

{'backbone': 'ResNet50', 'n_images': 374, 'n_patients': 213, 'accuracy': np.float64(0.7967914438502673), 'accuracy_ci_low': np.float64(0.7466648501362398), 'accuracy_ci_high': np.float64(0.8425233595800524), 'macro_f1': 0.7668635170603675, 'macro_f1_ci_low': np.float64(0.711534534818254), 'macro_f1_ci_high': np.float64(0.817575754956699), 'auc': np.float64(0.8406450815620367), 'auc_ci_low': np.float64(0.7820476600906594), 'auc_ci_high': np.float64(0.8913738678863476), 'misclassified': 76}
{'backbone': 'EfficientNetB0', 'n_images': 374, 'n_patients': 213, 'accuracy': np.float64(0.8315508021390374), 'accuracy_ci_low': np.float64(0.7828418230563002), 'accuracy_ci_high': np.float64(0.8733509234828496), 'macro_f1': 0.8111217103417316, 'macro_f1_ci_low': np.float64(0.7566861967380618), 'macro_f1_ci_high': np.float64(0.8569320796028974), 'auc': np.float64(0.8829708353929807), 'auc_ci_low': np.float64(0.8367229197935873), 'auc_ci_high': np.float64(0.9227257081273008), 'misclassified': 63}
{'ba

,backbone,n_images,n_patients,accuracy,accuracy_ci_low,accuracy_ci_high,macro_f1,macro_f1_ci_low,macro_f1_ci_high,auc,auc_ci_low,auc_ci_high,misclassified
0,ResNet50,374,213,0.7968,0.7467,0.8425,0.7669,0.7115,0.8176,0.8406,0.7820,0.8914,76
1,EfficientNetB0,374,213,0.8316,0.7828,0.8734,0.8111,0.7567,0.8569,0.8830,0.8367,0.9227,63
2,DenseNet121,374,213,0.8102,0.7631,0.8560,0.7888,0.7330,0.8386,0.8687,0.8192,0.9096,71


In [7]:
# 5. Table XI (held-out test partition of BUS-BRA)
f3 = lambda a, l, h: f"{a:.3f} [{l:.3f}, {h:.3f}]"
lines = [f"TABLE XI\nHELD-OUT TEST-SET CLASSIFICATION PERFORMANCE ON BUS-BRA ({N_IMG} IMAGES FROM {N_PAT} PATIENTS)\n",
         "| Backbone | Accuracy [95% CI] | Macro-F1 | AUC | Misclassified |", "|---|---|---|---|---|"]
for _, r in clf.iterrows():
    lines.append(f"| {r.backbone} | {f3(r.accuracy, r.accuracy_ci_low, r.accuracy_ci_high)} | {r.macro_f1:.3f} | {r.auc:.3f} | {r.misclassified} |")
lines.append(f"\nBecause the task is binary (benign vs malignant) there is no normal class. Macro-F1 is computed over benign and malignant; "
             f"AUC uses the malignant softmax probability with no tuned threshold. Confidence intervals use the cluster bootstrap over patients ({N_BOOT} resamples). "
             f"Training and test images come from the same dataset, so this is an internal held-out evaluation, not an external validation.")
text = "\n".join(lines)
open("/content/BUSBRA_table_xi.md", "w").write(text)
print(text)

TABLE XI
HELD-OUT TEST-SET CLASSIFICATION PERFORMANCE ON BUS-BRA (374 IMAGES FROM 213 PATIENTS)

| Backbone | Accuracy [95% CI] | Macro-F1 | AUC | Misclassified |
|---|---|---|---|---|
| ResNet50 | 0.797 [0.747, 0.843] | 0.767 | 0.841 | 76 |
| EfficientNetB0 | 0.832 [0.783, 0.873] | 0.811 | 0.883 | 63 |
| DenseNet121 | 0.810 [0.763, 0.856] | 0.789 | 0.869 | 71 |

Because the task is binary (benign vs malignant) there is no normal class. Macro-F1 is computed over benign and malignant; AUC uses the malignant softmax probability with no tuned threshold. Confidence intervals use the cluster bootstrap over patients (2000 resamples). Training and test images come from the same dataset, so this is an internal held-out evaluation, not an external validation.
